# IA propia de Viernes

Entrena los dos modelos propios de la app y los guarda en la carpeta de Drive:

1. **Intérprete neuronal** (`nlu_model.json`): entiende qué tarea, fecha, hora y repetición hay en una frase. Usa frases sintéticas y, si las hay, las exportadas por la app (`*.jsonl` en la carpeta de Drive).
2. **Palabra «Viernes»** (`viernes.tflite` + modelos de características): reemplaza a Vosk, pesa ~1 MB en vez de 38 MB y gasta menos batería.

Entorno de ejecución → Cambiar tipo → **GPU T4**. Luego Entorno de ejecución → **Ejecutar todo**.

In [ ]:
#@title 1. Código del proyecto
!nvidia-smi -L || echo 'Sin GPU: funciona igual, pero más lento'
!rm -rf /content/VIERNES && git clone -q --depth 1 https://github.com/ramdsoft-34/VIERNES.git /content/VIERNES
!cd /content/VIERNES && git log --oneline -1

In [ ]:
#@title 2. Dependencias
!pip install -q piper-tts audiomentations pyroomacoustics librosa soundfile onnxruntime-gpu
!pip install -q --no-deps openwakeword==0.6.0
import tensorflow as tf, sys
print('Python', sys.version.split()[0], '· TensorFlow', tf.__version__)

In [ ]:
#@title 3. Carpeta de salida en Drive
from google.colab import drive
drive.mount('/content/drive')
import os
OUT = '/content/drive/MyDrive/IA-Viernes/modelos'  #@param {type:'string'}
os.makedirs(OUT, exist_ok=True)
os.environ['OUT'] = OUT
print('Los modelos se guardan en', OUT)

In [ ]:
#@title 4. Intérprete neuronal (≈5 min)
!cd /content/VIERNES/training/nlu && python train_nlu.py --out "$OUT" --real "$OUT"/../*.jsonl "$OUT"/*.jsonl

In [ ]:
#@title 5. Palabra «Viernes» (≈40–60 min con GPU)
!cd /content/VIERNES/training/wake_word && python train_wakeword.py --out "$OUT" --phone-samples "$OUT"/../viernes-activaciones.zip

In [ ]:
#@title 6. Resultado
!ls -la "$OUT"
!cat "$OUT"/nlu_metrics.json "$OUT"/wakeword_metrics.json